In [1]:
# ==============================================================================
# PIPELINE 4: Master Merge & ML Dataset Preparation
# ==============================================================================
import pandas as pd
import numpy as np
from src.profiling import generate_minimal_report, generate_full_sample_report
from src.preparation.merging import (
    perform_master_merge, 
    compare_feature_consistency, 
    prepare_final_ml_dataset
)

# Global display settings for clean table outputs
pd.set_option('display.max_columns', None)

# ------------------------------------------------------------------------------
# STEP 1: Load the ML-Ready Lot Datasets
# ------------------------------------------------------------------------------
print("Loading cleaned lot-level datasets...")
df_cfc = pd.read_parquet("data/prepared/CFC_2018_2023.parquet")
df_can = pd.read_parquet("data/prepared/CAN_2018_2023.parquet")

print(f"CFC: {len(df_cfc):,} rows")
print(f"CAN: {len(df_can):,} rows")

Loading cleaned lot-level datasets...
CFC: 4,210,390 rows
CAN: 4,622,774 rows


In [2]:
# ------------------------------------------------------------------------------
# STEP 2: The Master Merge (Strict Inner Join on ID_LOT)
# ------------------------------------------------------------------------------
df_master = perform_master_merge(df_cfc, df_can)

# Quick sanity check of the first few rows
display(df_master.head())

Performing final Master Merge (Strict Inner Join on ID_LOT)...
Master Merge Complete! Total Rows: 4,623,976


,ID_NOTICE_CN,YEAR_CFC,ID_TYPE_CFC,FUTURE_CAN_ID,ISO_COUNTRY_CODE_CFC,B_MULTIPLE_COUNTRY_CFC,CAE_TYPE_CFC,MAIN_ACTIVITY_CFC,TYPE_OF_CONTRACT_CFC,CPV_CFC,ID_LOT_CFC,LOTS_NUMBER_CFC,B_EU_FUNDS_CFC,DURATION,TOP_TYPE_CFC,CRIT_CODE_CFC,B_RECURRENT_PROCUREMENT,TOTAL_VALUE_EUR_CFC,IS_FRAMEWORK_CFC,COOPERATIVE_PURCHASING_CFC,NUTS_REGION_COUNT_CFC,NUTS_LEVEL_CFC,PREPARATION_DAYS,JOIN_NOTICE,JOIN_LOT,ID_NOTICE_CAN,YEAR_CAN,ID_TYPE_CAN,ISO_COUNTRY_CODE_CAN,B_MULTIPLE_COUNTRY_CAN,CAE_TYPE_CAN,MAIN_ACTIVITY_CAN,TYPE_OF_CONTRACT_CAN,CPV_CAN,ID_LOT_CAN,LOTS_NUMBER_CAN,B_EU_FUNDS_CAN,TOP_TYPE_CAN,CRIT_CODE_CAN,NUMBER_AWARDS,ID_LOT_AWARDED,NUMBER_OFFERS,TOTAL_VALUE_EUR_CAN,LOT_AWARD_VALUE_EUR,IS_FRAMEWORK_CAN,COOPERATIVE_PURCHASING_CAN,NUMBER_OF_TENDERS,NUTS_REGION_COUNT_CAN,NUTS_LEVEL_CAN,DAYS_TO_AWARD,NUMBER_OF_WINNERS,NUMBER_WINNER_COUNTRIES,NUMBER_SME_WINNERS
0,20181,2018,2,2.022329e+09,BE,N,5,HEALTH,S,79,None,1.0,1.0,36.0,OPE,NaN,0.0,1.0,1,1,1,2.0,45.0,2022329163,NO_LOT_DEFINED,2022329163,2022,3,MT,N,3,OTHER,W,45,None,1.0,0.0,OPE,L,1,None,1.0,145492.63,145492.63,1,0,1.0,1,2.0,0.0,1,1,1
1,20182,2018,2,2.018490e+09,LU,N,5,OTHER,S,79,1,23.0,0.0,12.0,OPE,NaN,0.0,2400000.0,1,1,1,3.0,53.0,2018489569,1,2018489569,2018,3,LU,N,5,OTHER,S,79,1,23.0,0.0,OPE,M,23,None,NaN,411.92,NaN,1,0,NaN,1,3.0,NaN,0,0,0
2,20182,2018,2,2.018490e+09,LU,N,5,OTHER,S,79,10,23.0,0.0,12.0,OPE,NaN,0.0,2400000.0,1,1,1,3.0,53.0,2018489569,10,2018489569,2018,3,LU,N,5,OTHER,S,79,10,23.0,0.0,OPE,M,23,None,NaN,411.92,NaN,1,0,NaN,1,3.0,NaN,0,0,0
3,20182,2018,2,2.018490e+09,LU,N,5,OTHER,S,79,11,23.0,0.0,12.0,OPE,NaN,0.0,2400000.0,1,1,1,3.0,53.0,2018489569,11,2018489569,2018,3,LU,N,5,OTHER,S,79,11,23.0,0.0,OPE,M,23,None,NaN,411.92,NaN,1,0,NaN,1,3.0,NaN,0,0,0
4,20182,2018,2,2.018490e+09,LU,N,5,OTHER,S,79,12,23.0,0.0,12.0,OPE,NaN,0.0,2400000.0,1,1,1,3.0,53.0,2018489569,12,2018489569,2018,3,LU,N,5,OTHER,S,79,12,23.0,0.0,OPE,M,23,None,NaN,411.92,NaN,1,0,NaN,1,3.0,NaN,0,0,0


In [3]:
# ------------------------------------------------------------------------------
# STEP 3: Feature Consistency Check (Validation)
# ------------------------------------------------------------------------------
features_to_check = [
    'YEAR',
    'ID_TYPE',
    'ISO_COUNTRY_CODE',
    'CAE_TYPE',
    'MAIN_ACTIVITY',
    'CPV',
    'LOTS_NUMBER',
    'B_EU_FUNDS',
    'TOP_TYPE',
    'TOTAL_VALUE_EUR',
    'IS_FRAMEWORK',
    'COOPERATIVE_PURCHASING',
    'TYPE_OF_CONTRACT',
    'CRIT_CODE',
    'NUTS_REGION_COUNT',
    'NUTS_LEVEL'
]

consistency_df = compare_feature_consistency(df_master, features_to_check)
display(consistency_df)

Calculating Feature Consistency Rates...


,Feature,Match_Count,Total_Count,Consistency_Rate (%)
2,ISO_COUNTRY_CODE,4623101,4623976,99.98
12,TYPE_OF_CONTRACT,4613424,4623976,99.77
5,CPV,4601581,4623976,99.52
4,MAIN_ACTIVITY,4587812,4623976,99.22
8,TOP_TYPE,4580680,4623976,99.06
14,NUTS_REGION_COUNT,4573555,4623976,98.91
11,COOPERATIVE_PURCHASING,4573452,4623976,98.91
7,B_EU_FUNDS,4572602,4623976,98.89
3,CAE_TYPE,4564349,4623976,98.71
10,IS_FRAMEWORK,4556371,4623976,98.54


In [4]:
# ------------------------------------------------------------------------------
# STEP 4: Distill the Final Machine Learning Dataset
# ------------------------------------------------------------------------------
# This step removes Data Leakage features and isolates the target variables
df_ml_final = prepare_final_ml_dataset(df_master)

# Verify the final structure and columns
print("\nFinal columns in the ML dataset:")
print(df_ml_final.columns.tolist())
display(df_ml_final.head())


Preparing final ML Dataset (Selecting Features & Targets)...
 -> Merged ML Dataset shape: (4623976, 24)

Final columns in the ML dataset:
['ID_NOTICE', 'ID_LOT', 'NUMBER_OF_TENDERS', 'TARGET_AWARD_VALUE_EUR', 'LOT_AWARD_VALUE_EUR', 'YEAR', 'ID_TYPE', 'ISO_COUNTRY_CODE', 'CAE_TYPE', 'MAIN_ACTIVITY', 'CPV', 'LOTS_NUMBER', 'B_EU_FUNDS', 'DURATION', 'TOP_TYPE', 'B_RECURRENT_PROCUREMENT', 'ESTIMATED_VALUE_EUR', 'IS_FRAMEWORK', 'COOPERATIVE_PURCHASING', 'CRIT_CODE', 'TYPE_OF_CONTRACT', 'NUTS_REGION_COUNT', 'NUTS_LEVEL', 'PREPARATION_DAYS']


,ID_NOTICE,ID_LOT,NUMBER_OF_TENDERS,TARGET_AWARD_VALUE_EUR,LOT_AWARD_VALUE_EUR,YEAR,ID_TYPE,ISO_COUNTRY_CODE,CAE_TYPE,MAIN_ACTIVITY,CPV,LOTS_NUMBER,B_EU_FUNDS,DURATION,TOP_TYPE,B_RECURRENT_PROCUREMENT,ESTIMATED_VALUE_EUR,IS_FRAMEWORK,COOPERATIVE_PURCHASING,CRIT_CODE,TYPE_OF_CONTRACT,NUTS_REGION_COUNT,NUTS_LEVEL,PREPARATION_DAYS
0,0,0,1.0,145492.63,145492.63,2018,2,BE,5,HEALTH,79,1.0,1.0,36.0,OPE,0.0,1.0,1,1,NaN,S,1,2.0,45.0
1,1,1,NaN,411.92,NaN,2018,2,LU,5,OTHER,79,23.0,0.0,12.0,OPE,0.0,2400000.0,1,1,NaN,S,1,3.0,53.0
2,1,2,NaN,411.92,NaN,2018,2,LU,5,OTHER,79,23.0,0.0,12.0,OPE,0.0,2400000.0,1,1,NaN,S,1,3.0,53.0
3,1,3,NaN,411.92,NaN,2018,2,LU,5,OTHER,79,23.0,0.0,12.0,OPE,0.0,2400000.0,1,1,NaN,S,1,3.0,53.0
4,1,4,NaN,411.92,NaN,2018,2,LU,5,OTHER,79,23.0,0.0,12.0,OPE,0.0,2400000.0,1,1,NaN,S,1,3.0,53.0


In [5]:
# ------------------------------------------------------------------------------
# STEP 5: Save the "Golden Dataset" Checkpoint
# ------------------------------------------------------------------------------
output_path = "data/prepared/ted_combined.parquet"
df_ml_final.to_parquet(output_path, index=False)

# Generate a minimal report for the merged data
generate_minimal_report(
    df=df_ml_final,
    output_path="reports/merged/ted_combined_minimal.html",
    title="TED Combined (minimal; full data)"
)


# Generate a full report for a sample of the merged data
profiling_subset = df_ml_final.drop(columns=['ID_NOTICE', 'ID_LOT'], errors='ignore')
skewed_cols = ['NUMBER_OF_TENDERS', 'LOTS_NUMBER', 'DURATION', 'NUTS_LEVEL', 'PREPARATION_DAYS', 'LOT_AWARD_VALUE_EUR', 'TARGET_AWARD_VALUE_EUR', 'ESTIMATED_VALUE_EUR']
for col in skewed_cols:
    if col in profiling_subset.columns:
        profiling_subset[f"{col}_LOG1P"] = np.log1p(profiling_subset[col])
        # Drop the raw unscaled column from the report view to prevent crashes
        profiling_subset = profiling_subset.drop(columns=[col])

generate_full_sample_report(
    df=profiling_subset,
    output_path="reports/merged/ted_combined_full_sample.html",
    title="TED Combined (full; 250k sample)",
    sample_size=250000
)

Generating minimal report: 'TED Combined (minimal; full data)' ...


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 24/24 [00:01<00:00, 16.67it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

...

100%|██████████| 22/22 [00:00<00:00, 77.30it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

-> Full report successfully saved to: reports/merged/ted_combined_full_sample.html

